In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout,
                                     GlobalAveragePooling2D, BatchNormalization, Bidirectional, LSTM)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = ["./archive/fold{}/{}".format(row['fold'], row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Split into train and test sets
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Feature extraction function
def extract_features(file_path, n_mfcc=40, max_pad_len=40, n_fft=512):
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc, n_fft=n_fft)

        # Pad or truncate to ensure fixed shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.T  # Transpose for CNN input
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Apply feature extraction
X_train_features = np.array([f for f in (extract_features(fp) for fp in train_file_paths) if f is not None])
X_test_features = np.array([f for f in (extract_features(fp) for fp in test_file_paths) if f is not None])

# Apply PCA for dimensionality reduction
n_components = 40
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_features.reshape(X_train_features.shape[0], -1))
X_test_pca = pca.transform(X_test_features.reshape(X_test_features.shape[0], -1))

# Reshape for CNN input (batch_size, time_steps, features)
X_train_pca = X_train_pca.reshape(X_train_pca.shape[0], X_train_pca.shape[1], 1)
X_test_pca = X_test_pca.reshape(X_test_pca.shape[0], X_test_pca.shape[1], 1)

# Ensure y_train and y_test are properly formatted
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

# Callbacks for training
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)

# Function to build a 1D CNN model
def build_cnn_model():
    model = Sequential()

    # 1D CNN layers with Batch Normalization
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(n_components, 1)))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # Fully connected layers
    model.add(Flatten())  # Flatten before Dense layers
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(10, activation='softmax'))  # Ensure output shape is (None, 10)

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Function to build a Bi-ResNet-50 model
def build_biresnet_model():
    base_model = ResNet50(weights=None, include_top=False, input_shape=(40, 40, 3))
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(10, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Build models
cnn_model = build_cnn_model()
biresnet_model = build_biresnet_model()

# Train the CNN model
cnn_model.fit(X_train_pca, y_train, validation_data=(X_test_pca, y_test),
              epochs=30, batch_size=32, callbacks=[early_stopping, lr_scheduler])

# Fixing shape for ResNet50 input (Reshaping to (batch_size, 40, 40, 3))
X_train_resnet = X_train_pca.reshape(-1, 40, 40, 1)
X_train_resnet = np.repeat(X_train_resnet, 3, axis=-1)  # Convert 1-channel to 3-channel

X_test_resnet = X_test_pca.reshape(-1, 40, 40, 1)
X_test_resnet = np.repeat(X_test_resnet, 3, axis=-1)  # Convert 1-channel to 3-channel

# Train Bi-ResNet with reshaped input
biresnet_model.fit(X_train_resnet, y_train, validation_data=(X_test_resnet, y_test),
                   epochs=30, batch_size=32, callbacks=[early_stopping, lr_scheduler])

# Evaluate models
cnn_score = cnn_model.evaluate(X_test_pca, y_test)
biresnet_score = biresnet_model.evaluate(X_test_resnet, y_test)

print(f"CNN Model Accuracy: {cnn_score[1] * 100:.2f}%")
print(f"Bi-ResNet50 Model Accuracy: {biresnet_score[1] * 100:.2f}%")
